In [15]:
import pandas as pd

# Read a sample of the data
df = pd.read_parquet("green_tripdata_2025-11.parquet")

# Display first rows
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-11-01 00:34:48,2025-11-01 00:41:39,N,1.0,74,42,1.0,0.74,7.2,...,0.5,1.94,0.0,NaN,1.0,11.64,1.0,1.0,0.00,0.0
1,2,2025-11-01 00:18:52,2025-11-01 00:24:27,N,1.0,74,42,2.0,0.95,7.2,...,0.5,0.00,0.0,NaN,1.0,9.70,2.0,1.0,0.00,0.0
2,2,2025-11-01 01:03:14,2025-11-01 01:15:24,N,1.0,83,160,1.0,2.19,13.5,...,0.5,5.00,0.0,NaN,1.0,21.00,1.0,1.0,0.00,0.0
3,2,2025-11-01 00:10:57,2025-11-01 00:24:53,N,1.0,166,127,1.0,5.44,24.7,...,0.5,0.50,0.0,NaN,1.0,27.70,1.0,1.0,0.00,0.0
4,1,2025-11-01 00:03:48,2025-11-01 00:19:38,N,1.0,166,262,1.0,3.20,18.4,...,1.5,1.00,0.0,NaN,1.0,24.65,1.0,1.0,2.75,0.0


In [5]:
df.shape

(46912, 21)

In [6]:
df.dtypes

VendorID                          int32
lpep_pickup_datetime     datetime64[us]
lpep_dropoff_datetime    datetime64[us]
store_and_fwd_flag               object
RatecodeID                      float64
PULocationID                      int32
DOLocationID                      int32
passenger_count                 float64
trip_distance                   float64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
ehail_fee                       float64
improvement_surcharge           float64
total_amount                    float64
payment_type                    float64
trip_type                       float64
congestion_surcharge            float64
cbd_congestion_fee              float64
dtype: object

In [7]:
df["lpep_pickup_datetime"] = pd.to_datetime(df["lpep_pickup_datetime"])
df["lpep_pickup_datetime"]

0       2025-11-01 00:34:48
1       2025-11-01 00:18:52
2       2025-11-01 01:03:14
3       2025-11-01 00:10:57
4       2025-11-01 00:03:48
                ...        
46907   2025-11-30 19:58:34
46908   2025-11-30 19:34:00
46909   2025-11-30 21:46:46
46910   2025-11-30 21:00:00
46911   2025-11-30 23:26:00
Name: lpep_pickup_datetime, Length: 46912, dtype: datetime64[us]

In [10]:
count = (
    (df["lpep_pickup_datetime"] >= "2025-11-01") &
    (df["lpep_pickup_datetime"] < "2025-12-01") &
    (df["trip_distance"] <= 1)
).sum()

print(count)

8007


In [11]:
df_valid = df[df["trip_distance"] < 100]
df_valid.loc[df_valid["trip_distance"].idxmax()]

VendorID                                   2
lpep_pickup_datetime     2025-11-14 15:36:27
lpep_dropoff_datetime    2025-11-14 18:40:48
store_and_fwd_flag                         N
RatecodeID                               4.0
PULocationID                             130
DOLocationID                             265
passenger_count                          2.0
trip_distance                          88.03
fare_amount                            610.6
extra                                    0.0
mta_tax                                  0.5
tip_amount                               0.0
tolls_amount                             0.0
ehail_fee                                NaN
improvement_surcharge                    1.0
total_amount                           612.1
payment_type                             2.0
trip_type                                1.0
congestion_surcharge                     0.0
cbd_congestion_fee                       0.0
Name: 18867, dtype: object

In [12]:
zones = pd.read_csv("taxi_zone_lookup.csv")
zones.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [14]:
nov_18_trips = df[
    (df["lpep_pickup_datetime"] >= "2025-11-18") &
    (df["lpep_pickup_datetime"] < "2025-11-19")
]

# Aggregate total_amount by pickup location
amount_by_zone = (
    nov_18_trips
    .groupby("PULocationID", as_index=False)["total_amount"]
    .sum()
)

# Join with zone names
amount_by_zone = amount_by_zone.merge(
    zones,
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

# Find zone with the largest total_amount
top_zone = amount_by_zone.loc[amount_by_zone["total_amount"].idxmax()]
top_zone

PULocationID                   74
total_amount              9281.92
LocationID                     74
Borough                 Manhattan
Zone            East Harlem North
service_zone            Boro Zone
Name: 39, dtype: object

In [17]:
# Filter trips in November 2025
nov_trips = df[
    (df["lpep_pickup_datetime"] >= "2025-11-01") &
    (df["lpep_pickup_datetime"] < "2025-12-01")
]

# Merge pickup zones to get zone names
nov_trips = nov_trips.merge(
    zones[["LocationID", "Zone"]],
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
).rename(columns={"Zone": "pickup_zone"})

# Filter only trips picked up in "East Harlem North"
ehn_trips = nov_trips[nov_trips["pickup_zone"] == "East Harlem North"]

# Merge dropoff zones to get dropoff zone names
ehn_trips = ehn_trips.merge(
    zones[["LocationID", "Zone"]],
    left_on="DOLocationID",
    right_on="LocationID",
    how="left"
).rename(columns={"Zone": "dropoff_zone"})

# Find the row with the largest tip
max_tip_row = ehn_trips.loc[ehn_trips["tip_amount"].idxmax()]

# Extract results
dropoff_zone = max_tip_row["dropoff_zone"]
print("Dropoff zone with largest tip:", dropoff_zone)

Dropoff zone with largest tip: Yorkville West
